# Operations: Data Quality and Drift Report with Evidently

This notebook mirrors the **Data Drift & Model Decay** guide at [learnmlops.ops4life.com/guides/data-drift/](https://learnmlops.ops4life.com/guides/data-drift/).

Generate a comprehensive HTML monitoring report using Evidently: data drift, data quality, and target drift — all in a single shareable file.

**What we'll cover:**
1. Generate reference and current datasets
2. Define column roles
3. Build the monitoring report
4. Run the report
5. Save the HTML report
6. Extract a programmatic summary

In [ ]:
import pandas as pd
import numpy as np
import datetime
import os

os.makedirs('/workspace/pipeline-data', exist_ok=True)

## Step 1: Generate reference and current datasets

In [ ]:
np.random.seed(42)
reference = pd.DataFrame({
    'MonthlyIncome':     np.random.randint(1500, 20000, 1000).astype(float),
    'YearsAtCompany':    np.random.randint(0, 20, 1000).astype(float),
    'TotalWorkingYears': np.random.randint(0, 30, 1000).astype(float),
    'Age':               np.random.randint(22, 60, 1000).astype(float),
    'Department':        np.random.choice(['Sales', 'R&D', 'HR'], 1000, p=[0.30, 0.60, 0.10]),
    'OverTime':          np.random.choice(['Yes', 'No'], 1000, p=[0.28, 0.72]),
    'Attrition':         np.random.choice([0, 1], 1000, p=[0.84, 0.16]),
    'prediction':        np.random.choice([0, 1], 1000, p=[0.80, 0.20]),
})

np.random.seed(99)
current = pd.DataFrame({
    'MonthlyIncome':     np.random.randint(5000, 25000, 470).astype(float),
    'Age':               np.random.randint(22, 40, 470).astype(float),
    'YearsAtCompany':    np.random.randint(0, 20, 470).astype(float),
    'TotalWorkingYears': np.random.randint(0, 30, 470).astype(float),
    'Department':        np.random.choice(['Sales', 'R&D', 'HR'], 470, p=[0.45, 0.52, 0.03]),
    'OverTime':          np.random.choice(['Yes', 'No'], 470, p=[0.28, 0.72]),
    'Attrition':         np.random.choice([0, 1], 470, p=[0.84, 0.16]),
    'prediction':        np.random.choice([0, 1], 470, p=[0.75, 0.25]),
})

print(f'Reference: {reference.shape}  |  Current: {current.shape}')

## Step 2: Define column roles

In [ ]:
from evidently.pipeline.column_mapping import ColumnMapping

column_mapping = ColumnMapping(
    target='Attrition',
    prediction='prediction',
    numerical_features=['MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears', 'Age'],
    categorical_features=['Department', 'OverTime'],
)

print('Column mapping:')
print(f'  target:      {column_mapping.target}')
print(f'  prediction:  {column_mapping.prediction}')
print(f'  numeric:     {column_mapping.numerical_features}')
print(f'  categorical: {column_mapping.categorical_features}')

Column mapping tells Evidently which columns are features, targets, and predictions so each metric preset applies the right statistical test per column type.

## Step 3: Build the monitoring report

In [ ]:
from evidently.report import Report
from evidently.metric_presets import DataDriftPreset, DataQualityPreset, TargetDriftPreset

report = Report(metrics=[
    DataDriftPreset(),
    DataQualityPreset(),
    TargetDriftPreset(),
])

print('Report configured with:')
print('  - DataDriftPreset   (KS test per numeric feature, chi2 per categorical)')
print('  - DataQualityPreset (missing values, duplicates, out-of-range values)')
print('  - TargetDriftPreset (drift in the target/label distribution)')

## Step 4: Run the report

In [ ]:
report.run(
    reference_data=reference,
    current_data=current,
    column_mapping=column_mapping,
)
print('Report complete.')

## Step 5: Save the HTML report

In [ ]:
today = datetime.date.today().strftime('%Y-%m-%d')
html_path = f'/workspace/pipeline-data/drift_report_{today}.html'
report.save_html(html_path)
print(f'Saved: {html_path}')
print(f'Size:  {os.path.getsize(html_path):,} bytes')

Download the HTML file from the Jupyter file browser and open it in your browser to see the full interactive Evidently report. Share this file as evidence of model health during an incident review.

## Step 6: Extract a programmatic summary

In [ ]:
report_dict = report.as_dict()

# Navigate to the DataDrift section
drift_section = report_dict['metrics'][0]['result']
dataset_drift = drift_section.get('dataset_drift', False)
n_drifted     = drift_section.get('number_of_drifted_columns', 0)
n_columns     = drift_section.get('number_of_columns', 0)

print(f'Dataset drift detected: {dataset_drift}')
print(f'Drifted columns: {n_drifted} / {n_columns}')

if dataset_drift:
    print()
    print('ALERT: Drift threshold exceeded — consider scheduling a retraining run.')

## Next Steps

- Automate retraining with an Airflow DAG: `04-operations/data-drift-model-decay/retrain_dag.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/data-drift/](https://learnmlops.ops4life.com/guides/data-drift/)